In [ ]:
"""
Improved DCT Artifact Analysis for Deepfake Detection
Inspired by FreqNet paper's FFT analysis approach
"""

from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import dct, fft2, fftshift
from PIL import Image
from tqdm import tqdm

def dct2(x):
    """2D Discrete Cosine Transform"""
    return dct(dct(x.T, norm="ortho").T, norm="ortho")

def to_gray(img):
    """Convert image to grayscale numpy array"""
    if isinstance(img, Image.Image):
        img = img.convert("L")
        return np.array(img)
    return np.array(Image.fromarray(img).convert("L"))

def compute_mean_spectrum(images, method='dct', normalize=True):
    """
    Compute mean frequency spectrum across multiple images
    Similar to Figure 2 in the FreqNet paper
    
    Args:
        images: list of images
        method: 'dct' or 'fft'
        normalize: whether to normalize each spectrum before averaging
    """
    spectrums = []
    
    for img in tqdm(images, desc=f"Computing {method.upper()} spectrums"):
        # Convert to grayscale
        gray = to_gray(img)
        
        if method == 'dct':
            # DCT spectrum
            spectrum = dct2(gray)
        else:  # fft
            # FFT spectrum (shifted so DC is in center)
            spectrum = fftshift(fft2(gray))
        
        # Get magnitude
        spectrum_mag = np.abs(spectrum)
        
        # Optional normalization per image
        if normalize:
            spectrum_mag = spectrum_mag / (np.max(spectrum_mag) + 1e-8)
        
        spectrums.append(spectrum_mag)
    
    # Average across all images
    mean_spectrum = np.mean(spectrums, axis=0)
    
    # Log scale for better visualization
    mean_spectrum_log = np.log1p(mean_spectrum)
    
    return mean_spectrum_log

def plot_single_comparison(real_img, fake_img, method='dct'):
    """Plot single image comparison"""
    real_gray = to_gray(real_img)
    fake_gray = to_gray(fake_img)
    
    if method == 'dct':
        real_spec = np.log1p(np.abs(dct2(real_gray)))
        fake_spec = np.log1p(np.abs(dct2(fake_gray)))
        spec_name = "DCT"
    else:
        real_spec = np.log1p(np.abs(fftshift(fft2(real_gray))))
        fake_spec = np.log1p(np.abs(fftshift(fft2(fake_gray))))
        spec_name = "FFT"
    
    # Create difference map
    diff_spec = fake_spec - real_spec
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Real images
    axes[0, 0].imshow(real_gray, cmap="gray")
    axes[0, 0].set_title("Real Image", fontsize=12, fontweight='bold')
    axes[0, 0].axis("off")
    
    axes[0, 1].imshow(real_spec, cmap="hot")
    axes[0, 1].set_title(f"Real {spec_name} Spectrum", fontsize=12, fontweight='bold')
    axes[0, 1].axis("off")
    
    # Zoom into center for real
    center_crop_real = real_spec[96:160, 96:160]
    axes[0, 2].imshow(center_crop_real, cmap="hot")
    axes[0, 2].set_title(f"Real {spec_name} (Center Zoom)", fontsize=12, fontweight='bold')
    axes[0, 2].axis("off")
    
    # Fake images
    axes[1, 0].imshow(fake_gray, cmap="gray")
    axes[1, 0].set_title("Fake Image", fontsize=12, fontweight='bold')
    axes[1, 0].axis("off")
    
    axes[1, 1].imshow(fake_spec, cmap="hot")
    axes[1, 1].set_title(f"Fake {spec_name} Spectrum", fontsize=12, fontweight='bold')
    axes[1, 1].axis("off")
    
    # Zoom into center for fake
    center_crop_fake = fake_spec[96:160, 96:160]
    axes[1, 2].imshow(center_crop_fake, cmap="hot")
    axes[1, 2].set_title(f"Fake {spec_name} (Center Zoom)", fontsize=12, fontweight='bold')
    axes[1, 2].axis("off")
    
    plt.tight_layout()
    return fig

def plot_mean_comparison(ds, n_samples=100, method='dct'):
    """
    Create mean spectrum comparison like Figure 2 in FreqNet paper
    
    Args:
        ds: dataset
        n_samples: number of samples to average
        method: 'dct' or 'fft'
    """
    # Collect samples
    real_images = []
    fake_images = []
    
    for i, sample in enumerate(ds):
        if len(real_images) >= n_samples and len(fake_images) >= n_samples:
            break
        
        if sample["label"] == 0 and len(real_images) < n_samples:
            real_images.append(sample["image"])
        elif sample["label"] == 1 and len(fake_images) < n_samples:
            fake_images.append(sample["image"])
    
    print(f"\nComputing mean {method.upper()} spectrums...")
    print(f"Real images: {len(real_images)}, Fake images: {len(fake_images)}")
    
    # Compute mean spectrums
    mean_real = compute_mean_spectrum(real_images, method=method)
    mean_fake = compute_mean_spectrum(fake_images, method=method)
    
    # Compute difference
    difference = mean_fake - mean_real
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    im0 = axes[0].imshow(mean_real, cmap="hot")
    axes[0].set_title(f"Mean Real {method.upper()} Spectrum\n(avg of {len(real_images)} images)", 
                     fontsize=14, fontweight='bold')
    axes[0].axis("off")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    
    im1 = axes[1].imshow(mean_fake, cmap="hot")
    axes[1].set_title(f"Mean Fake {method.upper()} Spectrum\n(avg of {len(fake_images)} images)", 
                     fontsize=14, fontweight='bold')
    axes[1].axis("off")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    
    im2 = axes[2].imshow(difference, cmap="RdBu_r", vmin=-1, vmax=1)
    axes[2].set_title(f"Difference (Fake - Real)\n{method.upper()} Spectrum", 
                     fontsize=14, fontweight='bold')
    axes[2].axis("off")
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    return fig

def analyze_frequency_bands(spectrum):
    """
    Analyze energy distribution across frequency bands
    Similar to what FreqNet does with high-pass filtering
    """
    h, w = spectrum.shape
    
    # Define frequency bands (from low to high)
    # Band 1: Center (DC and very low freq) - 1/8 of image
    # Band 2: Low freq - 1/4 of image
    # Band 3: Mid freq - 1/2 of image
    # Band 4: High freq - rest
    
    center_y, center_x = h // 2, w // 2
    
    # Create masks for different frequency bands
    y, x = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((x - center_x)**2 + (y - center_y)**2)
    
    band1_mask = dist_from_center <= min(h, w) / 8   # DC + very low
    band2_mask = (dist_from_center > min(h, w) / 8) & (dist_from_center <= min(h, w) / 4)
    band3_mask = (dist_from_center > min(h, w) / 4) & (dist_from_center <= min(h, w) / 2)
    band4_mask = dist_from_center > min(h, w) / 2    # High freq
    
    # Calculate energy in each band
    energy_band1 = np.sum(spectrum[band1_mask])
    energy_band2 = np.sum(spectrum[band2_mask])
    energy_band3 = np.sum(spectrum[band3_mask])
    energy_band4 = np.sum(spectrum[band4_mask])
    
    total_energy = energy_band1 + energy_band2 + energy_band3 + energy_band4
    
    return {
        'DC_very_low': energy_band1 / total_energy * 100,
        'low': energy_band2 / total_energy * 100,
        'mid': energy_band3 / total_energy * 100,
        'high': energy_band4 / total_energy * 100
    }

def compare_frequency_distribution(ds, n_samples=50, method='dct'):
    """Compare frequency energy distribution between real and fake"""
    real_images = []
    fake_images = []
    
    for i, sample in enumerate(ds):
        if len(real_images) >= n_samples and len(fake_images) >= n_samples:
            break
        
        if sample["label"] == 0 and len(real_images) < n_samples:
            real_images.append(sample["image"])
        elif sample["label"] == 1 and len(fake_images) < n_samples:
            fake_images.append(sample["image"])
    
    # Compute mean spectrums
    mean_real = compute_mean_spectrum(real_images, method=method, normalize=False)
    mean_fake = compute_mean_spectrum(fake_images, method=method, normalize=False)
    
    # Analyze frequency bands
    real_dist = analyze_frequency_bands(np.exp(mean_real) - 1)  # Undo log transform
    fake_dist = analyze_frequency_bands(np.exp(mean_fake) - 1)
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    bands = list(real_dist.keys())
    real_values = list(real_dist.values())
    fake_values = list(fake_dist.values())
    
    x = np.arange(len(bands))
    width = 0.35
    
    ax1.bar(x - width/2, real_values, width, label='Real', alpha=0.8, color='blue')
    ax1.bar(x + width/2, fake_values, width, label='Fake', alpha=0.8, color='red')
    ax1.set_xlabel('Frequency Band', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Energy Distribution (%)', fontsize=12, fontweight='bold')
    ax1.set_title(f'Frequency Energy Distribution ({method.upper()})', 
                  fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(bands, rotation=15)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Difference plot
    differences = [fake_values[i] - real_values[i] for i in range(len(bands))]
    colors = ['green' if d < 0 else 'orange' for d in differences]
    
    ax2.bar(bands, differences, color=colors, alpha=0.7)
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    ax2.set_xlabel('Frequency Band', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Difference (Fake - Real) %', fontsize=12, fontweight='bold')
    ax2.set_title(f'Energy Difference by Band ({method.upper()})', 
                  fontsize=14, fontweight='bold')
    ax2.set_xticklabels(bands, rotation=15)
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    
    # Print numerical results
    print("\n" + "="*60)
    print(f"FREQUENCY ENERGY DISTRIBUTION ({method.upper()})")
    print("="*60)
    print(f"{'Band':<15} {'Real (%)':<12} {'Fake (%)':<12} {'Difference':<12}")
    print("-"*60)
    for band in bands:
        diff = fake_dist[band] - real_dist[band]
        print(f"{band:<15} {real_dist[band]:>10.2f}  {fake_dist[band]:>10.2f}  {diff:>+10.2f}")
    print("="*60)
    
    return fig

# Main execution
if __name__ == "__main__":
    print("Loading dataset...")
    ds = load_dataset("RohanRamesh/celebdfv2_small", split="train")
    
    print("\n1. Single Image Comparison (DCT)")
    real = next(x for x in ds if x["label"] == 0)
    fake = next(x for x in ds if x["label"] == 1)
    
    fig1 = plot_single_comparison(real["image"], fake["image"], method='dct')
    plt.show()  # Show plot
    fig1.savefig('single_comparison_dct.png', dpi=150, bbox_inches='tight')
    print("   Saved: single_comparison_dct.png")
    
    print("\n2. Single Image Comparison (FFT)")
    fig2 = plot_single_comparison(real["image"], fake["image"], method='fft')
    plt.show()  # Show plot
    fig2.savefig('single_comparison_fft.png', dpi=150, bbox_inches='tight')
    print("   Saved: single_comparison_fft.png")
    
    print("\n3. Mean Spectrum Comparison (DCT) - Like FreqNet Figure 2")
    fig3 = plot_mean_comparison(ds, n_samples=100, method='dct')
    plt.show()  # Show plot
    fig3.savefig('mean_comparison_dct.png', dpi=150, bbox_inches='tight')
    print("   Saved: mean_comparison_dct.png")
    
    print("\n4. Mean Spectrum Comparison (FFT)")
    fig4 = plot_mean_comparison(ds, n_samples=100, method='fft')
    plt.show()  # Show plot
    fig4.savefig('mean_comparison_fft.png', dpi=150, bbox_inches='tight')
    print("   Saved: mean_comparison_fft.png")
    
    print("\n5. Frequency Distribution Analysis (DCT)")
    fig5 = compare_frequency_distribution(ds, n_samples=50, method='dct')
    plt.show()  # Show plot
    fig5.savefig('frequency_distribution_dct.png', dpi=150, bbox_inches='tight')
    print("   Saved: frequency_distribution_dct.png")
    
    print("\n6. Frequency Distribution Analysis (FFT)")
    fig6 = compare_frequency_distribution(ds, n_samples=50, method='fft')
    plt.show()  # Show plot
    fig6.savefig('frequency_distribution_fft.png', dpi=150, bbox_inches='tight')
    print("   Saved: frequency_distribution_fft.png")
    
    print("\n✓ Analysis complete! All visualizations saved.")